In [1]:
!pip install transformers torch accelerate

   ---------------------------------------- 0.0/13.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.3 MB 6.1 MB/s eta 0:00:03
   ----- ---------------------------------- 1.8/13.3 MB 5.7 MB/s eta 0:00:03
   -------- ------------------------------- 2.9/13.3 MB 5.5 MB/s eta 0:00:02
   ------------ --------------------------- 4.2/13.3 MB 5.6 MB/s eta 0:00:02
   ---------------- ----------------------- 5.5/13.3 MB 5.9 MB/s eta 0:00:02
   ------------------- -------------------- 6.6/13.3 MB 5.8 MB/s eta 0:00:02
   ----------------------- ---------------- 7.9/13.3 MB 5.8 MB/s eta 0:00:01
   -------------------------- ------------- 8.9/13.3 MB 5.7 MB/s eta 0:00:01
   -------------------------------- ------- 10.7/13.3 MB 6.0 MB/s eta 0:00:01
   ----------------------------------- ---- 11.8/13.3 MB 6.0 MB/s eta 0:00:01
   ------------------------------------- -- 12.6/13.3 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 13.3/13.3 MB 5.8 MB/s  0:00:02
   --

In [1]:
!pip install -U jinja2

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {MODEL_ID} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(
    device
)
model.eval()

# Keep chat history across turns
chat_history = [{
    "role": "system",
    "content": (
        "You are a concise, helpful assistant for NIC e-Governance demos."
    ),
}]


def ask_bot(user_message: str, max_new_tokens: int = 150) -> str:
  chat_history.append({"role": "user", "content": user_message})

  prompt_text = tokenizer.apply_chat_template(
      chat_history, tokenize=False, add_generation_prompt=True
  )
  inputs = tokenizer([prompt_text], return_tensors="pt").to(device)

  with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Deterministic output for reliable live demos
        pad_token_id=tokenizer.eos_token_id,
    )

  # Slice off the input prompt tokens so only the new reply is decoded
  generated_ids = output_ids[0][inputs["input_ids"].shape[1] :]
  reply = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

  chat_history.append({"role": "assistant", "content": reply})
  return reply


print("Chatbot ready!")

Loading Qwen/Qwen2.5-0.5B-Instruct on cpu...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
print("User: Explain containerization in two short sentences.")
print("Bot :", ask_bot("Explain containerization in two short sentences."), "\n")

print("User: Give one example of how a district portal uses it.")
print("Bot :", ask_bot("Give one example of how a district portal uses it."))